# 三模型加权软投票的成员消融

使用 Notebook 022 导出的同一批验证集预测。从权重 (0.2, 0.3, 0.5) 的 B06、TextCNN、DistilBERT 完整融合出发，每次移除一个模型，按原比例把剩余权重归一化到和为 1。正类概率严格大于 0.5 时输出正类。

以完整加权融合为参照，统计每个双模型组合的 F1 变化、纠正的错误和新增的错误。另以三模型硬投票为共同基线，计算各组合保留了多少完整加权融合的 F1 增益。权重来自同一验证集上的选择，这次比较属于探索性分析。

In [ ]:
# 加载 Notebook 022 保存的概率，核对验证集样本及原有结果
import hashlib
import json

import numpy as np
import pandas as pd

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.paths import ROOT, CONFIGS_DIR, TABLES_DIR

protocol = json.loads((CONFIGS_DIR / "final_test_protocol.json").read_text(encoding="utf-8"))
validation_record = protocol["files"]["validation"]
validation_path = ROOT / validation_record["path"]
if hashlib.sha256(validation_path.read_bytes()).hexdigest() != validation_record["sha256"]:
    raise ValueError("Fixed validation split changed.")

labels = pd.read_csv(validation_path, usecols=["label"])["label"].to_numpy()
saved = pd.read_csv(TABLES_DIR / "three_model_soft_voting_validation_predictions.csv", float_precision="round_trip")
if (not np.array_equal(saved["VALIDATION ROW"], np.arange(len(labels)))
        or not np.array_equal(saved["LABEL"], labels)):
    raise ValueError("Saved predictions do not match the validation rows.")

model_names = ["B06", "TextCNN", "DistilBERT"]
probability_columns = [f"{name.upper()} POSITIVE PROBABILITY" for name in model_names]
probabilities = saved[probability_columns].to_numpy()
hard_vote = saved["Hard vote"].to_numpy()
if not np.isfinite(probabilities).all() or (probabilities < 0).any() or (probabilities > 1).any():
    raise ValueError("Invalid saved model probabilities.")
if not np.isin(hard_vote, [0, 1]).all():
    raise ValueError("Invalid hard vote predictions.")

weight_columns = [f"{name.upper()} WEIGHT" for name in model_names]
full_weights = np.array([0.2, 0.3, 0.5])
threshold = 0.5
full_vote = (probabilities @ full_weights > threshold).astype(int)
reference = pd.read_csv(TABLES_DIR / "three_model_soft_voting_validation.csv").set_index("MODEL")
full_reference = "Weighted soft vote (same validation)"
np.testing.assert_allclose(reference.loc[full_reference, weight_columns].to_numpy(dtype=float), full_weights)
baseline_metrics = {
    "Hard vote": classification_metrics(labels, hard_vote),
    full_reference: classification_metrics(labels, full_vote),
}
for name, metrics in baseline_metrics.items():
    np.testing.assert_allclose(reference.loc[name, list(metrics)].to_numpy(dtype=float),
                               list(metrics.values()), rtol=0, atol=1e-12)
print(f"Validation rows aligned: {len(labels):,}. Notebook 022 baselines reproduced.")

In [ ]:
# 每次移除一个模型并重新归一化，统计相对完整融合的错误变化
configurations = [
    ("Hard vote", None, hard_vote),
    ("Full weighted vote", full_weights, full_vote),
]
for index, name in enumerate(model_names):
    weights = full_weights.copy()
    weights[index] = 0
    weights /= weights.sum()
    prediction = (probabilities @ weights > threshold).astype(int)
    configurations.append((f"Without {name}", weights, prediction))

full_correct = full_vote == labels
full_f1 = baseline_metrics[full_reference]["F1"]
hard_f1 = baseline_metrics["Hard vote"]["F1"]
full_gain = full_f1 - hard_f1
rows = []
for name, weights, prediction in configurations:
    metrics = classification_metrics(labels, prediction)
    correct = prediction == labels
    gain = metrics["F1"] - hard_f1
    rows.append({
        "MODEL": name,
        **dict(zip(weight_columns, [None] * len(model_names) if weights is None else weights)),
        **metrics,
        "F1 CHANGE VS FULL (PP)": 100 * (metrics["F1"] - full_f1),
        "F1 CHANGE VS HARD (PP)": 100 * gain,
        "F1 GAIN RETAINED (%)": 100 * gain / full_gain,
        "CORRECTED VS FULL": int((~full_correct & correct).sum()),
        "NEW ERRORS VS FULL": int((full_correct & ~correct).sum()),
    })

ablation_results = pd.DataFrame(rows)
ablation_results.to_csv(TABLES_DIR / "three_model_voting_ablation_validation.csv", index=False)
display(ablation_results.round(4))

## 验证结果

| 方案 | F1 | 相对完整融合的 F1 变化 | 纠正完整融合的错误 | 新增错误 |
| --- | ---: | ---: | ---: | ---: |
| 完整加权融合 | 92.87% | — | — | — |
| 去掉 B06 | 92.17% | -0.69 个百分点 | 22 | 39 |
| 去掉 TextCNN | 91.59% | -1.27 个百分点 | 22 | 55 |
| 去掉 DistilBERT | 89.20% | -3.67 个百分点 | 38 | 129 |

B06 + DistilBERT 的权重分别为 2/7 和 5/7。其 F1 比三模型硬投票的 91.79% 低 0.20 个百分点，完整加权融合相对硬投票的 1.07 个百分点增益在这个双模型组合中没有保留。相对完整融合，该组合净增加 33 个错误。

去掉 B06 后，TextCNN + DistilBERT 是三个双模型组合里得分最高的，保留了约 35% 的 F1 增益。去掉 DistilBERT 时降幅最大。结果对应固定的原权重比例和阈值，三种双模型组合均未重新选权，也未在测试集上评估。
